# PHP2Shell: los avisos, los parches y las ramas sin soporte

Cuaderno de lectura de la medición `php2shell/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Las dos vulnerabilidades de PHP del 21 de octubre de 2026 (CVE-2026-109618 y CVE-2026-109619): la cronología con la fuente de cada hito, la demora del CVE al catálogo KEV comparada con las 1.758 entradas del catálogo, la hora del parche en Ubuntu, Red Hat, Debian, Alpine y la imagen oficial de Docker, la cuota de versiones de PHP en WordPress y la lista de productos afectados del NCSC-NL. Carga los ficheros de al lado (o los descarga del repositorio si se abre suelto).

In [ ]:
import json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/php2shell/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## Cronología

In [ ]:
cr = leer("cronologia.csv")
print(len(cr), "hitos ·", cr.tipo.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict())
cr[["fecha_utc", "hito", "actor", "tipo"]]

## Del CVE al KEV: la demora de PHP2Shell entre las 1.758 entradas

In [ ]:
k = leer("kev_demoras.csv")
rk = json.loads(texto("resumen_kev.json"))
print(len(k), "entradas · mediana", rk["todo"]["mediana"], "días · 2026:", rk["altas_2026"]["mediana"], "· desde la BOD 26-04:", rk["desde_bod_26_04"]["mediana"])
print("PHP2Shell:", rk["php2shell"])
k[k.cve.isin(["CVE-2026-109618", "CVE-2026-109619"])]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for etq, s in (("todo el catálogo", k.demora_dias), ("altas de 2026", k[k.kev_alta >= "2026-01-01"].demora_dias)):
    v = s.clip(lower=0.5).sort_values().values
    ax.step(v, range(1, len(v) + 1), where="post", label=etq)
ax.set_xscale("log"); ax.axvline(5, ls="--", color="green"); ax.set_xlabel("días del CVE al KEV"); ax.set_ylabel("entradas"); ax.legend(); plt.show()

## El parche en cada sistema

In [ ]:
s = leer("parches_sistemas.csv")
print(len(s), "filas ·", s.sistema.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict())
s.groupby("sistema").horas_desde_parche.agg(["min", "max"]).sort_values("min")

## Versiones de PHP en WordPress

In [ ]:
w = leer("wordpress_php.csv")
rw = json.loads(texto("resumen_wordpress.json"))
print(rw["grupos"])
w.plot.bar(x="version", y="cuota_pct", legend=False, figsize=(8, 3.5), color=["tab:green" if g.startswith("con") else ("tab:orange" if g.startswith("solo") else "tab:red") for g in w.grupo]); plt.ylabel("% de los WordPress"); plt.show()

## Productos con PHP dentro (NCSC-NL)

In [ ]:
n = leer("ncsc_productos.csv")
rn = json.loads(texto("resumen_ncsc.json"))
print(rn["productos"], "productos de", rn["proveedores"], "proveedores ·", rn["estados_109618"])
n.estado_109618.value_counts().sort_index().sort_values(ascending=False, kind="stable")